#Problem: Support Ticket Chat Tracker

Build a LangGraph app that handles customer messages on support tickets. Each ticket is one thread, and the graph must remember the conversation and some counters across multiple invoke calls.

In [ ]:
from typing import TypedDict,Annotated,Literal
import operator

billing = ["refund","payment"]
technical = ["crash","error"]
class MessageState(TypedDict):
    message : Annotated[list[str],operator.add]
    category : Literal["billing","technical","general"]
    turn_cnt : int

def classify(state : MessageState):
    msg = state["message"][-1].lower()
    if state.get("category","") == "":
        if any(word in msg for word in billing):
            category = "billing"
        elif any(word in msg for word in technical):
            category = "technical"
        else :
            category = "general"
        return {"category":category}
    return {"category":state["category"]}
    
def respond(state:MessageState):
    return{
        "message":[f"{state['category']} # {state.get('turn_cnt',0)}:we have got your issue,looking into it. "]
    }

def count(state:MessageState):
    cnt = state.get("turn_cnt",0) +1
    return {
        "turn_cnt" : cnt
    }

In [ ]:
from langgraph.graph import StateGraph,START, END
from langgraph.checkpoint.memory import InMemorySaver

graph = StateGraph(state_schema=MessageState)

# add nodes
graph.add_node("classify",classify)
graph.add_node("respond",respond)
graph.add_node("count",count)

# add edges
graph.add_edge(START,"classify")
graph.add_edge("classify","respond")
graph.add_edge("respond","count")
graph.add_edge("count",END)

memory = InMemorySaver()
workflow = graph.compile(checkpointer=memory)

config1 = {"configurable":{"thread_id":"1"}}



In [ ]:
ticket1 = workflow.invoke({"message":["Product is not good, refund neeed."]},config=config1)
ticket1 = workflow.invoke({"message":["Still investigating?"]},config=config1)
ticket1 = workflow.invoke({"message":["Any progress?"]},config=config1)

print(ticket1)

In [ ]:
workflow.get_state(config = config1) # Prove InMemory is temporary. Restart the kernel (or rerun the script) and call get_state on ticket-101. Explain what you see.
# got empty snapshot - StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
config2 = {"configurable":{"thread_id":"2"}}

ticket2 = workflow.invoke({"message":["System has error."]},config=config2)
ticket2 = workflow.invoke({"message":["Need fix on priority."]},config=config2)

print(ticket2)

In [ ]:
workflow.get_state(config=config1).values

In [ ]:
workflow.get_state(config=config1).metadata

In [ ]:
workflow.get_state(config=config1).next # next is empty since it has reached already end node

Predict, then check the history. Before running get_state_history on ticket-202, write down how many checkpoints you expect. Then print step, source, and next for each, and compare with your prediction. -> Total 15 , each invoke - 5 snapshots (2 comes from start itself)

In [ ]:
for s in workflow.get_state_history(config=config1):
    print(f"{s.metadata['step']}***{s.metadata['source']} *** {s.values} *** {s.next}**{s.values.get('turn_cnt')}")

In [ ]:
for s in workflow.get_state_history(config=config1):
    print(s.config)

Delete a ticket. Call checkpointer.delete_thread("ticket-202") and confirm that ticket-101 is unaffected.

In [ ]:
memory.delete_thread("2")

In [ ]:
workflow.get_state(config = config1).values["category"]

In [ ]:
workflow.get_state(config = config2) # already deleted, so gives empty snapshot.

# update

In [ ]:
workflow.update_state(config=config1, values ={"category":"general"})

In [ ]:
workflow.get_state(config=config1).values["category"]

In [ ]:
workflow.invoke({"message":["test update thread"]},config=config1)

# time travel :
older checkpoint and fork from there

In [ ]:
workflow.get_state_history(config=config1)

In [ ]:
oldTip = workflow.get_state(config=config1).config
oldTip

In [ ]:
history = list(workflow.get_state_history(config=config1))

# find the checkpoint where the first invoke had finished
first_done = [s for s in history if s.values.get("turn_cnt")==1 and s.next == ()][0]

fork_config = workflow.update_state(config=first_done.config,values = {"category":"technical"},as_node="classify")
workflow.invoke({"message":["Fork config testing"]},config = fork_config)



In [ ]:
for s in workflow.get_state_history(config=config1):
    print(s)

1f1c9f66-00f7-66e8-8013-8cc6b4402a73

# Note : 
The fix: skip update_state and invoke from the snapshot’s config directly.
update_state goes through the reducers, just like a node’s return value. Your message field uses operator.add, so passing the snapshot’s whole message list appends it again:

In [ ]:
# oldBranchConfigShot = [s for s in workflow.get_state_history(config=config1) if s.config['configurable']['checkpoint_id']=="1f1c9f66-00f7-66e8-8013-8cc6b4402a73"][0]

# oldFork = workflow.update_state(config = oldBranchConfigShot.config,values = oldBranchConfigShot.values)

# workflow.invoke(config = oldFork,input={"message":["Hello"]})

# Getting old branch using update state

In [ ]:
history = list(workflow.get_state_history(config=config1))

oldTip = [s for s in history if s.values['message'][-2:-1] == ['test update thread'] and s.next == ()][0].config
oldTip

In [ ]:
workflow.update_state(config=oldTip,values={})

In [ ]:
for s in workflow.get_state_history(config=config1):
    print(s)

# replay from past checkpoint

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import time, random

class Support(TypedDict):
    msg1 : str
    msg2 : str
    msg3 : str

def node1(state:Support):
    msg1 = "1st node testing"
    return { "msg1":msg1}

def node2(state:Support):
    msg2 = f"2nd node testing {random.random()}"
    return { "msg2":msg2}

def node3(state:Support):
    msg3 = "3rd node testing"
    return { "msg3":msg3}

build = StateGraph(state_schema=Support)
build.add_node("node1",node1)
build.add_node("node2",node2)
build.add_node("node3",node3)

build.add_edge(START,"node1")
build.add_edge("node1","node2")
build.add_edge("node2","node3")
build.add_edge("node3",END)

checkpointer = InMemorySaver()
workflow = build.compile(checkpointer=checkpointer)
workflow.invoke({},config= {"configurable":{"thread_id":"thread123"}})



In [ ]:
for s in workflow.get_state_history(config= {"configurable":{"thread_id":"thread123"}}):
    print(s.config, s.next)

In [ ]:
history = list(workflow.get_state_history(config= {"configurable":{"thread_id":"thread123"}}))

# get node of node2 point 
node2Config = [s for s in history if s.next == ("node2",)][0]

In [ ]:
node2Config.config

In [ ]:
workflow.invoke(None,config=node2Config.config)